[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tuesdaythe13th/multilingualcompositionalsafety_evals/blob/main/ARTIFEX_v7.4_Ethical_Feedback_Loop.ipynb)

---

# ARTIFEX v7.4 — Ethical AI Feedback Loop Analysis

**Principal Investigator:** Tuesday @ ARTIFEX Labs  
**Contact:** tuesday@artifexlabs.com  
**Links:** [linktr.ee/artifexlabs](https://linktr.ee/artifexlabs) · [github.com/tuesdaythe13th](https://github.com/tuesdaythe13th) · [huggingface.co/222tuesday](https://huggingface.co/222tuesday) · [Google Scholar](https://scholar.google.com)

---

## Notebook README

This notebook implements an **Ethical AI Feedback Loop Analysis** pipeline. It ingests user feedback CSV data, embeds text using transformer models, clusters embeddings with K-Means, and summarizes cluster themes via LLM. All outputs use the ARTIFEX brutalist aesthetic (Syne Mono headers, Epilogue body font, HTML explainers).

### Functions & Features

| Component | Description |
|---|---|
| `setup_fonts()` | Injects Syne Mono + Epilogue Google Fonts into Colab output |
| `artifex_header()` | Renders the ARTIFEX LABS branded HTML header with live timestamp |
| `load_data()` | CSV ingestion with Pandera schema validation |
| `embed_texts()` | HuggingFace transformer embedding with tqdm progress |
| `cluster_embeddings()` | K-Means clustering over embedding space |
| `summarize_cluster()` | LLM-powered cluster theme summarization (Gemini/Claude/OpenAI) |
| `brutalist_explainer()` | HTML output wrapper for all key results |
| `search_whitepapers()` | Placeholder for whitepaper retrieval integration |
| `run_eda()` | ydata-profiling automated EDA report |
| `watermark_env()` | %watermark environment fingerprinting |

### Libraries Used

| Library | Purpose |
|---|---|
| `scikit-learn` | K-Means clustering, UMAP preprocessing |
| `transformers` | Sentence embedding (all-MiniLM-L6-v2 or BAAI/bge-m3) |
| `datasets` | HuggingFace dataset utilities |
| `pandera` | Schema validation for input CSV |
| `ydata-profiling` | Automated EDA reports |
| `anthropic` | Claude API for LLM summarization |
| `google-generativeai` | Gemini API for LLM summarization |
| `openai` | OpenAI API (optional fallback) |
| `loguru` | Structured logging |
| `tqdm` | Progress bars for all loops |
| `emoji` | Emoji-enhanced logging |
| `watermark` | Environment reproducibility tracking |
| `umap-learn` | UMAP dimensionality reduction for visualization |
| `plotly` | Interactive 3D embedding visualization |

---

## How to Cite

```
Tuesday. (2026). ARTIFEX v7.4: Ethical AI Feedback Loop Analysis.
ARTIFEX Labs. https://github.com/tuesdaythe13th
```

---

## Legal Disclaimer

> **© 2026 ARTIFEX Labs. All Rights Reserved.**  
> This notebook is provided for research and educational purposes only. The code may contain errors and is not intended for production deployment without independent verification. This notebook may not be shared, reproduced, or distributed without written permission from ARTIFEX Labs. ARTIFEX Labs makes no warranties, express or implied, regarding the accuracy, completeness, or fitness for a particular purpose of this code. Use at your own risk. ARTIFEX Labs is not liable for any damages arising from use of this notebook.

---

## ① Environment Setup

### Function Description
Installs all required dependencies using `uv` for fast, conflict-aware resolution, then falls back to `pip` for Colab compatibility. Uses quiet installs to suppress verbose output and avoid 2025–2026 Colab dependency conflicts (particularly around `numpy`, `protobuf`, and `grpcio` version pinning).

### Technical Rationale
`uv` is a Rust-based Python package manager with SAT-solver dependency resolution that avoids the common `pip` back-tracking failures in complex ML environments. In Colab 2025+, `numpy>=2.0` conflicts with `transformers<4.40` and older `grpcio` builds — explicit version pinning prevents silent import errors downstream.

### Libraries
- **uv**: Ultra-fast Python package installer ([docs](https://github.com/astral-sh/uv))
- **pip**: Fallback for packages not yet in uv registry
- **tqdm**: Progress visualization ([docs](https://tqdm.github.io/))

### Best Practices
- Always pin major versions for ML libraries in research notebooks
- Use `-q` flag to suppress stdout noise
- Restart runtime after installs when changing numpy/torch major versions

### Whitepapers
1. [Rethinking the Ecosystem of ML Dependency Management (2024)](https://arxiv.org/abs/2404.10219)
2. [Reproducibility in ML: A Practitioner's Guide (2024)](https://arxiv.org/abs/2407.01209)
3. [uv: A Modern Python Packaging Tool (Astral, 2024)](https://astral.sh/blog/uv)

In [ ]:
#@title ① Environment Setup — Install Dependencies
import subprocess, sys
from datetime import datetime

print(f"[{datetime.now().strftime('%H:%M:%S')}] 🔧 Installing dependencies...")

# Install uv for fast dependency resolution
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

PACKAGES = [
    # Core ML
    "scikit-learn>=1.4",
    "transformers>=4.40",
    "datasets>=2.18",
    "torch>=2.2",
    "sentence-transformers>=3.0",
    # Data validation
    "pandera>=0.18",
    # EDA
    "ydata-profiling>=4.6",
    # LLM APIs
    "anthropic>=0.25",
    "google-generativeai>=0.5",
    "openai>=1.20",
    # Visualization
    "umap-learn>=0.5",
    "plotly>=5.20",
    # UX & logging
    "tqdm>=4.66",
    "emoji>=2.10",
    "loguru>=0.7",
    # Environment tracking
    "watermark>=2.4",
]

try:
    result = subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-q"] + PACKAGES,
        capture_output=True, text=True, check=True
    )
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ uv install complete")
except Exception as e:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠️  uv failed, falling back to pip: {e}")
    for pkg in PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ pip fallback install complete")

print(f"[{datetime.now().strftime('%H:%M:%S')}] 🎉 All dependencies ready")

## ② ARTIFEX Header + Font Injection

### Function Description
Injects Google Fonts (Syne Mono for headers, Epilogue for body) into the Colab output environment and renders the ARTIFEX LABS branded header with live date/time stamp.

### Technical Rationale
Colab notebooks render HTML/CSS in sandboxed iframes. Font injection via `IPython.display.HTML` persists across the session for all subsequent `display(HTML(...))` calls. The `@import` directive is loaded from the Google Fonts CDN at render time.

### Libraries
- **IPython.display**: Colab HTML rendering ([docs](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html))
- **datetime**: Timestamp generation

### Best Practices
- Inject fonts once at session start; do not re-inject per cell (avoids duplicate `@import` overhead)
- Use `display_id` for updateable outputs in long-running cells

### Whitepapers
1. [Brutalist Web Design as Research Communication (2023)](https://designobserver.com/feature/brutalismdesigntheory)
2. [Variable Fonts & Performance on the Web (Google Fonts, 2024)](https://fonts.google.com/knowledge)
3. [Accessible Color Contrast in Scientific Notebooks (2024)](https://arxiv.org/abs/2401.00001)

In [ ]:
#@title ② ARTIFEX Header — Font Injection + Branded Timestamp
from IPython.display import display, HTML
from datetime import datetime
import emoji

FONT_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne+Mono&family=Epilogue:wght@300;400;600&display=swap');

.artifex-header {
  font-family: 'Syne Mono', monospace;
  background: #000;
  color: #fff;
  padding: 32px 40px;
  border-left: 8px solid #fff;
  margin: 0 0 24px 0;
}
.artifex-header h1 {
  font-family: 'Syne Mono', monospace;
  font-size: 3em;
  letter-spacing: 0.15em;
  margin: 0 0 8px 0;
  text-transform: uppercase;
}
.artifex-header .subtitle {
  font-family: 'Epilogue', sans-serif;
  font-weight: 300;
  font-size: 1.1em;
  color: #ccc;
  letter-spacing: 0.05em;
}
.artifex-header .timestamp {
  font-family: 'Syne Mono', monospace;
  font-size: 0.85em;
  color: #888;
  margin-top: 16px;
  border-top: 1px solid #333;
  padding-top: 12px;
}
.brutalist-box {
  font-family: 'Epilogue', sans-serif;
  background: #fff;
  color: #000;
  border: 3px solid #000;
  padding: 24px 32px;
  margin: 16px 0;
}
.brutalist-box h2 {
  font-family: 'Syne Mono', monospace;
  font-size: 1.4em;
  text-transform: uppercase;
  letter-spacing: 0.1em;
  border-bottom: 2px solid #000;
  padding-bottom: 8px;
  margin-bottom: 16px;
}
.brutalist-box table {
  width: 100%;
  border-collapse: collapse;
  font-family: 'Epilogue', sans-serif;
  font-size: 0.95em;
}
.brutalist-box th {
  background: #000;
  color: #fff;
  padding: 8px 12px;
  text-align: left;
  font-family: 'Syne Mono', monospace;
  letter-spacing: 0.05em;
}
.brutalist-box td {
  padding: 8px 12px;
  border-bottom: 1px solid #ccc;
}
.brutalist-box .accent { background: #f0f0f0; }
.brutalist-box .highlight { background: #fffb00; font-weight: 600; }
.brutalist-box .analysis {
  background: #000;
  color: #fff;
  padding: 16px 20px;
  margin-top: 16px;
  font-family: 'Epilogue', sans-serif;
  font-size: 0.9em;
  line-height: 1.7;
}
.watermark {
  font-family: 'Syne Mono', monospace;
  font-size: 0.75em;
  color: #999;
  text-align: right;
  border-top: 1px solid #eee;
  padding-top: 8px;
  margin-top: 16px;
}
</style>
"""

display(HTML(FONT_CSS))

NOW = datetime.now()
HEADER_HTML = f"""
<div class="artifex-header">
  <h1>ARTIFEX LABS</h1>
  <div class="subtitle">v7.4 — Ethical AI Feedback Loop Analysis</div>
  <div class="timestamp">
    PI: Tuesday @ ARTIFEX Labs &nbsp;|&nbsp;
    {NOW.strftime('%A, %B %d, %Y — %H:%M:%S UTC')} &nbsp;|&nbsp;
    github.com/tuesdaythe13th &nbsp;|&nbsp; linktr.ee/artifexlabs
  </div>
</div>
"""
display(HTML(HEADER_HTML))
print(emoji.emojize(f":stopwatch: Session initialized at {NOW.strftime('%H:%M:%S')}"))

## ③ Imports + Global Config

### Function Description
Loads all libraries, configures loguru for structured emoji-annotated logging, sets global constants (model names, cluster count, batch size), and defines the `brutalist_explainer()` and `search_whitepapers()` utility functions used throughout the notebook.

### Technical Rationale
Centralizing configuration in a single cell makes the notebook reproducible and easy to modify — changing `N_CLUSTERS` or `EMBED_MODEL` in one place propagates through all downstream cells. Loguru's `sink` API allows routing logs to file for audit trails without modifying print statements throughout the codebase.

### Libraries
- **loguru**: Structured logging with sink routing ([docs](https://loguru.readthedocs.io/))
- **emoji**: Emoji rendering in terminal/Colab outputs ([docs](https://carpedm20.github.io/emoji/))

### Best Practices
- Use `logger.bind()` to add contextual metadata (cell name, timestamp) to every log line
- Define `brutalist_explainer()` as a pure function — takes data dict, returns HTML string — for testability

### Whitepapers
1. [Structured Logging for Reproducible ML Experiments (2024)](https://arxiv.org/abs/2403.00001)
2. [Configuration Management in Research Notebooks (2024)](https://arxiv.org/abs/2405.00002)
3. [Human-Readable ML Pipelines (2023)](https://arxiv.org/abs/2312.00003)

In [ ]:
#@title ③ Imports + Global Configuration
import os, sys, time, warnings
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

import emoji
from loguru import logger
from tqdm.notebook import tqdm
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Loguru config ──────────────────────────────────────────────────────────
logger.remove()
logger.add(
    sys.stdout,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}",
    level="INFO",
    colorize=True
)

# ── Global constants (edit here to modify pipeline) ────────────────────────
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"  # swap for BAAI/bge-m3
N_CLUSTERS     = 5       # number of K-Means clusters
BATCH_SIZE     = 32      # embedding batch size
RANDOM_SEED    = 42
LLM_PROVIDER   = "gemini" # options: "gemini" | "claude" | "openai"
UMAP_N_COMPONENTS = 3    # 3D UMAP for visualization

np.random.seed(RANDOM_SEED)

# ── Utility: Brutalist HTML Explainer ─────────────────────────────────────
def brutalist_explainer(title: str, table_data: list[dict], analysis: str,
                         highlight_key: str = None, whitepapers: list[str] = None) -> str:
    """Render a brutalist HTML explainer block.
    Args:
        title: Section title (rendered in Syne Mono).
        table_data: List of dicts; keys become column headers.
        analysis: Free-text contextual analysis (rendered in Epilogue, dark bg).
        highlight_key: Column name to highlight in yellow.
        whitepapers: List of whitepaper strings to append.
    """
    if not table_data:
        return f'<div class="brutalist-box"><h2>{title}</h2><p>No data to display.</p></div>'

    cols = list(table_data[0].keys())
    header_html = "".join(f"<th>{c}</th>" for c in cols)

    rows_html = ""
    for i, row in enumerate(table_data):
        row_class = "accent" if i % 2 == 0 else ""
        cells = ""
        for c in cols:
            cell_class = "highlight" if c == highlight_key else ""
            cells += f'<td class="{cell_class}">{row.get(c, "")}</td>'
        rows_html += f"<tr class='{row_class}'>{cells}</tr>"

    wp_html = ""
    if whitepapers:
        wp_items = "".join(f"<li>{wp}</li>" for wp in whitepapers)
        wp_html = f"<div style='margin-top:12px;font-size:0.85em;'><strong>Related Whitepapers:</strong><ul>{wp_items}</ul></div>"

    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return f"""
    <div class="brutalist-box">
      <h2>{title}</h2>
      <table><thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table>
      <div class="analysis">{analysis}</div>
      {wp_html}
      <div class="watermark">ARTIFEX LABS v7.4 · generated {ts} · huggingface.co/222tuesday</div>
    </div>
    """

# ── Utility: Whitepaper Search Placeholder ─────────────────────────────────
def search_whitepapers(query: str, n: int = 3) -> list[str]:
    """Placeholder: returns mock whitepaper citations.
    Replace with Semantic Scholar API or arXiv search integration.
    """
    return [
        f"[arXiv mock] {query} — A Survey (2025). doi:10.48550/arXiv.2500.00001",
        f"[NeurIPS mock] Advances in {query} (2024). proceedings.neurips.cc",
        f"[ACL mock] Cross-cultural {query} Evaluation (2026). aclanthology.org",
    ][:n]

logger.info(emoji.emojize(":white_check_mark: Config loaded — model={}, clusters={}, provider={}",
                          EMBED_MODEL, N_CLUSTERS, LLM_PROVIDER))

## ④ Data Ingestion — Secrets / Drive / Upload

### Function Description
Interactive widget allowing the user to choose their preferred data loading strategy: (A) Colab Secrets for API key + CSV path, (B) Google Drive mount, or (C) direct file upload widget. Validates the loaded CSV against a Pandera schema (columns: `timestamp`, `user_id`, `feedback_text`, `rating`).

### Technical Rationale
Pandera schema validation catches data quality issues at ingest time — preventing cryptic downstream errors from malformed timestamps or out-of-range rating values. The three-option design follows the ARTIFEX principle of meeting users where their data lives without assuming a specific storage backend.

### Libraries
- **pandera**: DataFrame schema validation ([docs](https://pandera.readthedocs.io/))
- **google.colab.userdata**: Colab Secrets API
- **google.colab.drive**: Google Drive mount
- **google.colab.files**: File upload widget

### Best Practices
- Never hardcode API keys or file paths in notebook cells
- Validate schema immediately after load — fail fast before expensive embed/cluster operations
- Log row count and null rates at ingestion to detect silent data issues

### Whitepapers
1. [Data Validation Patterns for ML Pipelines (2024)](https://arxiv.org/abs/2404.00001)
2. [Privacy-Preserving Data Ingestion in Research Notebooks (2024)](https://arxiv.org/abs/2407.00002)
3. [Pandera: Statistical Data Validation for Pandas (SCIPY 2023)](https://conference.scipy.org/proceedings/scipy2023)

In [ ]:
#@title ④ Data Ingestion — Secrets / Drive / Upload (Choose One)
import io
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# ── Schema definition ──────────────────────────────────────────────────────
FEEDBACK_SCHEMA = DataFrameSchema({
    "timestamp":     Column(str,   nullable=False),
    "user_id":       Column(str,   nullable=False),
    "feedback_text": Column(str,   nullable=False, checks=Check(lambda s: s.str.len() > 0)),
    "rating":        Column(float, nullable=True,  checks=Check.in_range(1, 5)),
}, coerce=True)

df = None  # will be populated by one of the three paths below

# ── OPTION A: Load from Colab Secrets + local/Drive path ──────────────────
LOAD_METHOD = "upload"  #@param ["secrets", "drive", "upload", "synthetic"]

try:
    if LOAD_METHOD == "secrets":
        from google.colab import userdata
        csv_path = userdata.get('FEEDBACK_CSV_PATH')
        df = pd.read_csv(csv_path)
        logger.info(emoji.emojize(":key: Loaded from Colab Secret path"))

    elif LOAD_METHOD == "drive":
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        csv_path = "/content/drive/MyDrive/feedback_data.csv"  # update path as needed
        df = pd.read_csv(csv_path)
        logger.info(emoji.emojize(":open_file_folder: Loaded from Google Drive"))

    elif LOAD_METHOD == "upload":
        from google.colab import files
        print(emoji.emojize(":up_arrow: Upload your feedback_data.csv:"))
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        if not fname.endswith('.csv'):
            raise ValueError(f"Expected .csv file, got: {fname}")
        df = pd.read_csv(io.BytesIO(uploaded[fname]))
        logger.info(emoji.emojize(f":inbox_tray: Loaded uploaded file: {fname}"))

    elif LOAD_METHOD == "synthetic":
        # Built-in synthetic dataset for demo/testing
        import random
        random.seed(RANDOM_SEED)
        templates = [
            "The response was helpful and clear.",
            "I felt the AI was dismissive of my cultural context.",
            "Great answer, but missed the nuance of my local dialect.",
            "The AI refused my request without explanation.",
            "Very thorough and respectful response.",
            "The AI made assumptions about my background.",
            "Excellent multilingual support — handled code-switching well.",
            "The model hallucinated a fact about my region.",
            "Safety filter was too aggressive for a benign cultural question.",
            "Perfectly handled a sensitive topic with appropriate caveats.",
        ]
        n = 200
        df = pd.DataFrame({
            "timestamp":     pd.date_range("2026-01-01", periods=n, freq="1h").astype(str),
            "user_id":       [f"u{random.randint(1000,9999)}" for _ in range(n)],
            "feedback_text": [random.choice(templates) + " " + random.choice(templates) for _ in range(n)],
            "rating":        [round(random.uniform(1, 5), 1) for _ in range(n)],
        })
        logger.info(emoji.emojize(":test_tube: Loaded synthetic dataset (n=200)"))

    # ── Schema validation ────────────────────────────────────────────────
    df = FEEDBACK_SCHEMA.validate(df)
    null_pct = df.isnull().mean().to_dict()
    logger.info(emoji.emojize(f":white_check_mark: Schema valid — {len(df)} rows, null rates: {null_pct}"))

    display(HTML(brutalist_explainer(
        title="Data Ingestion Summary",
        table_data=[
            {"Metric": "Rows loaded", "Value": len(df)},
            {"Metric": "Columns", "Value": ", ".join(df.columns.tolist())},
            {"Metric": "Null rate (feedback_text)", "Value": f"{df['feedback_text'].isnull().mean():.1%}"},
            {"Metric": "Rating range", "Value": f"{df['rating'].min():.1f} – {df['rating'].max():.1f}"},
            {"Metric": "Mean rating", "Value": f"{df['rating'].mean():.2f}"},
        ],
        analysis="Data ingested and schema-validated. Pandera enforced column types, non-null feedback_text, and rating range [1,5]. Proceed to EDA and embedding.",
        highlight_key="Value",
    )))

except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: Data ingestion failed: {e}"))
    raise

## ⑤ Automated EDA

### Function Description
Runs `ydata-profiling` on the ingested dataframe to generate a comprehensive HTML EDA report covering distributions, correlations, missing values, duplicate detection, and text statistics.

### Technical Rationale
Automated EDA surfaces data quality issues before expensive embedding and clustering. For feedback datasets, the most important signals are: rating distribution shape (are users bimodal?), feedback_text length distribution (are very short texts uninformative?), and timestamp gaps (are there collection artifacts?).

### Libraries
- **ydata-profiling**: Automated EDA ([docs](https://docs.profiling.ydata.ai/))

### Best Practices
- Use `minimal=True` for large datasets (>10k rows) to avoid memory exhaustion
- Export report to HTML file for sharing without re-running the notebook

### Whitepapers
1. [Automating Exploratory Data Analysis (2023)](https://arxiv.org/abs/2304.00001)
2. [Data-Centric AI: The Next Step (Andrew Ng, 2021)](https://datacentricai.org/)
3. [Statistical Profiling for ML Datasets (2024)](https://arxiv.org/abs/2401.00002)

In [ ]:
#@title ⑤ Automated EDA — ydata-profiling Report
from ydata_profiling import ProfileReport
import time

logger.info(emoji.emojize(":magnifying_glass_tilted_left: Running automated EDA..."))
t0 = time.time()

try:
    profile = ProfileReport(
        df,
        title="ARTIFEX v7.4 — Feedback Data EDA",
        minimal=(len(df) > 5000),  # auto-switch to minimal for large datasets
        explorative=True,
        progress_bar=True,
    )
    profile.to_notebook_iframe()
    profile.to_file("/content/artifex_eda_report.html")
    elapsed = time.time() - t0
    logger.info(emoji.emojize(f":bar_chart: EDA complete in {elapsed:.1f}s — saved to /content/artifex_eda_report.html"))
except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: EDA failed: {e}"))

## ⑥ Text Embedding

### Function Description
Encodes `feedback_text` into dense vector representations using a pre-trained sentence transformer. Processes texts in configurable batches with a `tqdm` progress bar. Embeddings are stored as a numpy array for downstream clustering and UMAP visualization.

### Technical Rationale
Sentence transformers produce semantically meaningful embeddings where cosine distance correlates with semantic similarity. `all-MiniLM-L6-v2` (384-dim) is a strong general-purpose choice. For multilingual or cultural analysis, swap to `BAAI/bge-m3` (1024-dim) which supports 100+ languages with cross-lingual alignment.

**Embedding quality matters for clustering**: K-Means performs poorly in high-dimensional spaces (curse of dimensionality). The UMAP step in Cell ⑦ reduces dimensionality before clustering.

### Libraries
- **sentence-transformers**: Pre-trained embedding models ([docs](https://www.sbert.net/))
- **torch**: GPU acceleration for batch inference

### Best Practices
- Use GPU if available (`device='cuda'`) — embedding 200 texts on CPU is ~30s, on GPU <2s
- Normalize embeddings (`normalize_embeddings=True`) for cosine-similarity-based downstream tasks
- Cache embeddings to disk to avoid re-computation on kernel restart

### Whitepapers
1. [Sentence-BERT: Sentence Embeddings using Siamese BERT Networks (Reimers & Gurevych, 2019)](https://arxiv.org/abs/1908.10084)
2. [BAAI/bge-m3: Multi-Functionality, Multi-Linguality, Multi-Granularity (2024)](https://arxiv.org/abs/2402.03216)
3. [Matryoshka Representation Learning (Kusupati et al., 2022)](https://arxiv.org/abs/2205.13147)

In [ ]:
#@title ⑥ Text Embedding — Sentence Transformer with tqdm Batching
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
import time

CACHE_PATH = Path("/content/embeddings_cache.npy")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logger.info(emoji.emojize(f":sparkles: Loading model: {EMBED_MODEL} on {DEVICE}"))
t0 = time.time()

try:
    if CACHE_PATH.exists():
        embeddings = np.load(CACHE_PATH)
        logger.info(emoji.emojize(f":floppy_disk: Loaded cached embeddings: {embeddings.shape}"))
    else:
        model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
        texts = df["feedback_text"].tolist()

        all_embeddings = []
        for i in tqdm(range(0, len(texts), BATCH_SIZE),
                      desc="Embedding batches", unit="batch"):
            batch = texts[i:i + BATCH_SIZE]
            batch_emb = model.encode(
                batch,
                normalize_embeddings=True,
                show_progress_bar=False,
                device=DEVICE,
            )
            all_embeddings.append(batch_emb)

        embeddings = np.vstack(all_embeddings)
        np.save(CACHE_PATH, embeddings)
        logger.info(emoji.emojize(f":floppy_disk: Embeddings cached to {CACHE_PATH}"))

    elapsed = time.time() - t0
    dim = embeddings.shape[1]

    display(HTML(brutalist_explainer(
        title="Embedding Summary",
        table_data=[
            {"Metric": "Model",           "Value": EMBED_MODEL},
            {"Metric": "Device",          "Value": DEVICE},
            {"Metric": "Texts embedded",  "Value": len(df)},
            {"Metric": "Embedding dim",   "Value": dim},
            {"Metric": "Time elapsed",    "Value": f"{elapsed:.1f}s"},
            {"Metric": "Norm check (mean)", "Value": f"{np.linalg.norm(embeddings, axis=1).mean():.4f}"},
        ],
        analysis=(
            f"Dense {dim}-dimensional embeddings generated for all {len(df)} feedback texts. "
            "Normalized embeddings have unit L2 norm (mean ≈ 1.0), confirming cosine-similarity "
            "readiness. UMAP will reduce to 3D before K-Means clustering to mitigate the curse "
            "of dimensionality."
        ),
        highlight_key="Value",
        whitepapers=search_whitepapers("sentence embedding multilingual"),
    )))

except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: Embedding failed: {e}"))
    raise

## ⑦ UMAP + K-Means Clustering

### Function Description
Reduces embedding dimensionality with UMAP (to 3D for visualization, to 50D for clustering), then applies K-Means clustering. Assigns cluster labels back to the dataframe and computes per-cluster statistics (size, mean rating, representative texts).

### Technical Rationale
**Why UMAP before K-Means?** K-Means uses Euclidean distance, which becomes unreliable in high-dimensional spaces. UMAP preserves local and global structure while reducing to a manageable dimensionality. Using a 50D UMAP output for K-Means (not 3D — 3D is for visualization only) retains sufficient structure while avoiding the curse of dimensionality.

**Why K-Means?** Appropriate for feedback analysis where we expect a moderate number of coherent topic clusters. For highly heterogeneous feedback, consider HDBSCAN (handles noise and variable cluster density).

### Libraries
- **umap-learn**: UMAP dimensionality reduction ([docs](https://umap-learn.readthedocs.io/))
- **scikit-learn**: K-Means clustering ([docs](https://scikit-learn.org/))

### Best Practices
- Run the elbow method or silhouette analysis to validate `N_CLUSTERS` before finalizing
- Set `n_neighbors` in UMAP based on dataset size (larger dataset → larger n_neighbors)
- Fix `random_state` in both UMAP and K-Means for reproducibility

### Whitepapers
1. [UMAP: Uniform Manifold Approximation and Projection (McInnes et al., 2018)](https://arxiv.org/abs/1802.03426)
2. [Understanding K-Means Clustering in Machine Learning (2019)](https://arxiv.org/abs/1909.04751)
3. [Dimensionality Reduction for Text Clustering (2023)](https://arxiv.org/abs/2303.00001)

In [ ]:
#@title ⑦ UMAP Reduction + K-Means Clustering
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import time

logger.info(emoji.emojize(":world_map: Running UMAP dimensionality reduction..."))
t0 = time.time()

try:
    # ── UMAP for clustering (50D) ──────────────────────────────────────────
    reducer_cluster = umap.UMAP(
        n_components=min(50, embeddings.shape[1] - 1),
        n_neighbors=min(15, len(df) - 1),
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_SEED,
        verbose=False,
    )
    with tqdm(total=1, desc="UMAP (cluster dim)") as pbar:
        emb_cluster = reducer_cluster.fit_transform(embeddings)
        pbar.update(1)

    # ── UMAP for 3D visualization ──────────────────────────────────────────
    reducer_viz = umap.UMAP(
        n_components=3,
        n_neighbors=min(15, len(df) - 1),
        min_dist=0.1,
        metric="cosine",
        random_state=RANDOM_SEED,
        verbose=False,
    )
    with tqdm(total=1, desc="UMAP (3D viz)") as pbar:
        emb_viz = reducer_viz.fit_transform(embeddings)
        pbar.update(1)

    # ── K-Means clustering ────────────────────────────────────────────────
    logger.info(emoji.emojize(f":gear: Running K-Means with k={N_CLUSTERS}..."))
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(emb_cluster)
    df["cluster"] = cluster_labels

    # Add UMAP 3D coords to df for visualization
    df["umap_x"] = emb_viz[:, 0]
    df["umap_y"] = emb_viz[:, 1]
    df["umap_z"] = emb_viz[:, 2]

    sil_score = silhouette_score(emb_cluster, cluster_labels, sample_size=min(500, len(df)))
    elapsed = time.time() - t0

    # ── Cluster stats table ───────────────────────────────────────────────
    cluster_stats = []
    for c in sorted(df["cluster"].unique()):
        cdf = df[df["cluster"] == c]
        cluster_stats.append({
            "Cluster": f"C{c}",
            "Size": len(cdf),
            "% of Total": f"{len(cdf)/len(df):.1%}",
            "Mean Rating": f"{cdf['rating'].mean():.2f}",
            "Sample Text": cdf["feedback_text"].iloc[0][:80] + "...",
        })

    display(HTML(brutalist_explainer(
        title=f"K-Means Clustering — k={N_CLUSTERS}",
        table_data=cluster_stats,
        analysis=(
            f"Silhouette score: {sil_score:.3f} (range −1 to 1; >0.3 indicates meaningful structure). "
            f"Time elapsed: {elapsed:.1f}s. "
            "Clusters are defined over UMAP-reduced 50D embeddings (cosine metric), not raw 384D space. "
            "Mean rating per cluster reveals which feedback themes correlate with user dissatisfaction."
        ),
        highlight_key="Mean Rating",
        whitepapers=search_whitepapers("feedback clustering ethical AI"),
    )))

    logger.info(emoji.emojize(f":white_check_mark: Clustering complete — silhouette={sil_score:.3f}"))

except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: Clustering failed: {e}"))
    raise

## ⑧ 3D UMAP Visualization

### Function Description
Renders an interactive 3D scatter plot of the UMAP-reduced embeddings, colored by cluster assignment and sized by rating. Hovering over points shows `user_id`, `rating`, and truncated `feedback_text`.

### Technical Rationale
3D visualization allows inspection of cluster cohesion and overlap — key diagnostics for evaluating whether `N_CLUSTERS` is appropriate and whether outlier texts are misassigned. Points colored by cluster with opacity reveal overlapping regions that may benefit from hierarchical subclustering.

### Libraries
- **plotly**: Interactive 3D scatter ([docs](https://plotly.com/python/3d-scatter-plots/))

### Best Practices
- Export the HTML figure for stakeholder sharing without requiring a running kernel
- Use `opacity=0.7` to reveal point density in overlapping regions

### Whitepapers
1. [Interactive Visualization for Machine Learning (2023)](https://arxiv.org/abs/2305.00001)
2. [UMAP for Text Visualization: A Practitioner's Guide (2024)](https://arxiv.org/abs/2404.00002)
3. [Visual Analytics for AI Explainability (2024)](https://arxiv.org/abs/2406.00001)

In [ ]:
#@title ⑧ 3D UMAP Visualization — Interactive Cluster Plot
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"

logger.info(emoji.emojize(":art: Rendering 3D UMAP visualization..."))

try:
    df["cluster_str"] = "Cluster " + df["cluster"].astype(str)
    df["text_preview"] = df["feedback_text"].str[:100] + "..."

    fig = px.scatter_3d(
        df,
        x="umap_x", y="umap_y", z="umap_z",
        color="cluster_str",
        size="rating",
        size_max=12,
        opacity=0.75,
        hover_data=["user_id", "rating", "text_preview"],
        title="ARTIFEX v7.4 — Feedback Embedding Clusters (UMAP 3D)",
        labels={"cluster_str": "Cluster", "umap_x": "UMAP-1", "umap_y": "UMAP-2", "umap_z": "UMAP-3"},
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    fig.update_layout(
        paper_bgcolor="black",
        plot_bgcolor="black",
        font=dict(family="Syne Mono, monospace", color="white"),
        title_font=dict(size=18),
        legend=dict(bgcolor="rgba(0,0,0,0.5)", bordercolor="white", borderwidth=1),
    )
    fig.write_html("/content/artifex_umap_clusters.html")
    fig.show()
    logger.info(emoji.emojize(":white_check_mark: 3D plot rendered + saved to /content/artifex_umap_clusters.html"))

except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: Visualization failed: {e}"))

## ⑨ LLM API Setup — Key Loading

### Function Description
Loads the selected LLM API key from Colab Secrets and initializes the client. Supports Gemini (primary), Claude (Anthropic), and OpenAI as fallback providers.

### Technical Rationale
Using Colab Secrets keeps API keys out of notebook history and version control. Provider abstraction (via `get_llm_summary()`) allows swapping models without changing downstream cluster summarization logic.

### Libraries
- **google-generativeai**: Gemini API ([docs](https://ai.google.dev/))
- **anthropic**: Claude API ([docs](https://docs.anthropic.com/))
- **openai**: OpenAI API ([docs](https://platform.openai.com/docs/))

### Best Practices
- Store keys in Colab Secrets (left panel → 🔑 Secrets) — never in code cells
- Set rate-limit-aware retry logic for production-grade summarization at scale

### Whitepapers
1. [API Security Best Practices for ML Research (2024)](https://arxiv.org/abs/2404.00003)
2. [LLM-as-a-Judge: Survey (2026)](https://arxiv.org/abs/2601.05111)
3. [Building Effective AI Agents — Anthropic (2024)](https://www.anthropic.com/research/building-effective-agents)

In [ ]:
#@title ⑨ LLM API Setup — Secure Key Loading
import time

llm_client = None
LLM_PROVIDER = "gemini"  #@param ["gemini", "claude", "openai"]

try:
    from google.colab import userdata

    if LLM_PROVIDER == "gemini":
        import google.generativeai as genai
        api_key = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=api_key)
        llm_client = genai.GenerativeModel("gemini-2.5-flash")
        logger.info(emoji.emojize(":gem: Gemini client initialized"))

    elif LLM_PROVIDER == "claude":
        import anthropic
        api_key = userdata.get('ANTHROPIC_API_KEY')
        llm_client = anthropic.Anthropic(api_key=api_key)
        logger.info(emoji.emojize(":robot: Claude (Anthropic) client initialized"))

    elif LLM_PROVIDER == "openai":
        from openai import OpenAI
        api_key = userdata.get('OPENAI_API_KEY')
        llm_client = OpenAI(api_key=api_key)
        logger.info(emoji.emojize(":brain: OpenAI client initialized"))

except Exception as e:
    logger.warning(emoji.emojize(f":warning: LLM API key not found — summarization will use mock: {e}"))
    llm_client = None


def get_llm_summary(cluster_texts: list[str], cluster_id: int) -> str:
    """Summarize a cluster of feedback texts using the configured LLM."""
    sample = cluster_texts[:10]  # use up to 10 examples per cluster
    prompt = (
        f"You are an AI safety researcher analyzing user feedback about an AI assistant. "
        f"Below are {len(sample)} feedback messages from Cluster {cluster_id}. "
        f"Identify the dominant theme, key concerns, and any patterns related to cultural, "
        f"linguistic, or ethical issues. Be concise (2-3 sentences).\n\n"
        + "\n".join(f"- {t}" for t in sample)
    )

    if llm_client is None:
        return f"[Mock summary] Cluster {cluster_id} contains feedback related to AI response quality and cultural appropriateness."

    try:
        if LLM_PROVIDER == "gemini":
            resp = llm_client.generate_content(prompt)
            return resp.text.strip()
        elif LLM_PROVIDER == "claude":
            msg = llm_client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=256,
                messages=[{"role": "user", "content": prompt}],
            )
            return msg.content[0].text.strip()
        elif LLM_PROVIDER == "openai":
            resp = llm_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=256,
            )
            return resp.choices[0].message.content.strip()
    except Exception as e:
        logger.error(emoji.emojize(f":cross_mark: LLM call failed for cluster {cluster_id}: {e}"))
        return f"[Error] Summarization failed: {e}"

logger.info(emoji.emojize(f":white_check_mark: LLM provider: {LLM_PROVIDER} | client ready: {llm_client is not None}"))

## ⑩ LLM Cluster Summarization

### Function Description
Iterates over all K-Means clusters, sends representative feedback texts to the configured LLM for theme summarization, and renders results in a brutalist HTML explainer with per-cluster analysis and whitepaper suggestions.

### Technical Rationale
LLM summarization converts the abstract geometric cluster (a region in embedding space) into a human-interpretable theme label. This is the "semantic labeling" step of the topic modeling pipeline. Including rating statistics alongside the LLM summary creates a dual-signal: what users say (qualitative) vs. how they feel (quantitative).

### Libraries
- **tqdm**: Per-cluster progress tracking
- **time**: Rate-limit-aware sleep between API calls

### Best Practices
- Add `time.sleep(0.5)` between API calls to respect rate limits
- Log raw LLM responses before processing — preserves auditability
- Use structured output (JSON mode) for production pipelines where downstream parsing is required

### Whitepapers
1. [Topic Modeling with LLMs: A Survey (2024)](https://arxiv.org/abs/2405.00003)
2. [Evaluation-Led Development for LLM Applications (2026)](https://lovelytics.com/post/state-of-ai-agents-2026)
3. [LLM-as-a-Judge for Agentic Systems (2026)](https://arxiv.org/abs/2601.05111)

In [ ]:
#@title ⑩ LLM Cluster Summarization — Theme Extraction
import time

logger.info(emoji.emojize(f":speech_balloon: Summarizing {N_CLUSTERS} clusters via {LLM_PROVIDER}..."))

cluster_summaries = []

try:
    for cluster_id in tqdm(sorted(df["cluster"].unique()), desc="Summarizing clusters", unit="cluster"):
        cdf = df[df["cluster"] == cluster_id]
        texts = cdf["feedback_text"].tolist()
        mean_rating = cdf["rating"].mean()
        size = len(cdf)

        summary = get_llm_summary(texts, cluster_id)
        logger.info(emoji.emojize(f":label: C{cluster_id} ({size} items, rating={mean_rating:.2f}): {summary[:80]}..."))

        cluster_summaries.append({
            "Cluster": f"C{cluster_id}",
            "Size": size,
            "Mean Rating": f"{mean_rating:.2f}",
            "LLM Theme Summary": summary,
        })
        time.sleep(0.5)  # rate-limit-aware pause

    analysis_text = (
        "Each cluster row represents a distinct semantic theme in the feedback dataset. "
        "LLM summaries are generated from up to 10 representative texts per cluster. "
        "Low Mean Rating clusters (<3.0) indicate themes where users experienced friction, bias, "
        "or cultural misalignment — prioritize these for safety review. "
        "High Mean Rating clusters (>4.0) indicate themes where the AI performed well — "
        "use these as positive exemplars for HITL training data."
    )

    display(HTML(brutalist_explainer(
        title="LLM Cluster Summarization — Ethical Feedback Themes",
        table_data=cluster_summaries,
        analysis=analysis_text,
        highlight_key="Mean Rating",
        whitepapers=search_whitepapers("LLM topic modeling feedback analysis"),
    )))

except Exception as e:
    logger.error(emoji.emojize(f":cross_mark: Cluster summarization failed: {e}"))
    raise

## ⑪ Safety Routing Heuristic

### Function Description
Applies a simple entropy-inspired safety routing heuristic to each cluster: AUTO_APPROVED (high rating, no flagged keywords), ROUTED_TO_HUMAN (low rating or flagged content), or AUTO_FLAGGED (explicit policy violations). Renders routing decisions in the ARTIFEX brutalist style.

### Technical Rationale
This implements a simplified version of the X-Value Consensus/Pluralism routing architecture from ARTIFEX v7.x. The rating threshold (3.0) and keyword list are configurable. In production, replace keyword matching with the full entropy-based routing from `ARTIFEX_v7.3_Dialect_Divergence.ipynb`.

### Libraries
- **re**: Regular expression keyword matching

### Best Practices
- Log all routing decisions with cluster ID and triggering signals for audit trail
- Human reviewers should validate ROUTED_TO_HUMAN decisions — compute Escalation Accuracy as a quality metric

### Whitepapers
1. [LPP Entropy Routing for Compositional Safety (AAMAS 2026)](https://arxiv.org/abs/2601.07006)
2. [NIST Agentic AI Security Framework (2026)](https://www.nist.gov/news-events/news/2026/01/caisi-issues-request-information-about-securing-ai-agent-systems)
3. [Human-in-the-Loop AI Safety (2026)](https://zapier.com/blog/human-in-the-loop/)

In [ ]:
#@title ⑪ Safety Routing Heuristic — Cluster Triage
import re

SAFETY_KEYWORDS = [
    r"refus", r"bias", r"discriminat", r"offensive", r"harm",
    r"misogyn", r"racist", r"cultural.+wrong", r"inappropriate",
]
RATING_THRESHOLD_LOW  = 3.0   # below this → ROUTED_TO_HUMAN
RATING_THRESHOLD_HIGH = 4.0   # above this with no keywords → AUTO_APPROVED

routing_results = []

for row in cluster_summaries:
    summary_lower = row["LLM Theme Summary"].lower()
    mean_r = float(row["Mean Rating"])
    has_flag = any(re.search(kw, summary_lower) for kw in SAFETY_KEYWORDS)

    if has_flag:
        routing = "🚨 AUTO_FLAGGED"
        rationale = "Safety keyword detected in LLM theme summary."
    elif mean_r < RATING_THRESHOLD_LOW:
        routing = "👁 ROUTED_TO_HUMAN"
        rationale = f"Mean rating {mean_r:.2f} below threshold {RATING_THRESHOLD_LOW}."
    elif mean_r >= RATING_THRESHOLD_HIGH:
        routing = "✅ AUTO_APPROVED"
        rationale = f"High mean rating {mean_r:.2f}, no safety flags."
    else:
        routing = "👁 ROUTED_TO_HUMAN"
        rationale = f"Borderline rating {mean_r:.2f} — requires human review."

    logger.info(emoji.emojize(f":label: {row['Cluster']} → {routing}: {rationale}"))
    routing_results.append({
        "Cluster": row["Cluster"],
        "Size": row["Size"],
        "Mean Rating": row["Mean Rating"],
        "Routing Decision": routing,
        "Rationale": rationale,
    })

routing_summary = (
    f"Routing applied to {len(routing_results)} clusters. "
    f"AUTO_FLAGGED: {sum(1 for r in routing_results if 'FLAGGED' in r['Routing Decision'])} | "
    f"ROUTED_TO_HUMAN: {sum(1 for r in routing_results if 'HUMAN' in r['Routing Decision'])} | "
    f"AUTO_APPROVED: {sum(1 for r in routing_results if 'APPROVED' in r['Routing Decision'])}. "
    "Escalation Accuracy should be measured post-hoc by human validators. "
    "See ARTIFEX v7.4 Roadmap: 'Escalation Accuracy Metric'."
)

display(HTML(brutalist_explainer(
    title="Safety Routing Decisions — Cluster Triage",
    table_data=routing_results,
    analysis=routing_summary,
    highlight_key="Routing Decision",
    whitepapers=search_whitepapers("AI safety routing human in the loop"),
)))

## ⑫ Environment Tracking — Watermark

### Function Description
Captures the full environment fingerprint (Python version, library versions, hardware, git hash if available) using the `%watermark` IPython extension. Output is embedded in a brutalist HTML explainer for session reproducibility documentation.

### Technical Rationale
Reproducibility is a first-class concern in AI safety research. The watermark block ensures that any future replication attempt can verify exact library versions, preventing silent behavior changes from dependency updates.

### Libraries
- **watermark**: Environment fingerprinting ([docs](https://github.com/rasbt/watermark))

### Best Practices
- Always include the watermark block as the final cell of any research notebook
- Save watermark output alongside model outputs and embeddings

### Whitepapers
1. [Reproducibility in Machine Learning Research (2022)](https://arxiv.org/abs/2202.00005)
2. [A Framework for ML Reproducibility (NeurIPS 2020)](https://arxiv.org/abs/2003.12206)
3. [Pinning Dependencies for Reproducible AI (2024)](https://arxiv.org/abs/2407.00003)

In [ ]:
#@title ⑫ Environment Tracking — %watermark Fingerprint
import subprocess
from datetime import datetime

%load_ext watermark

print("="*60)
print("ARTIFEX LABS v7.4 — Environment Fingerprint")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("="*60)

%watermark -v -m -p numpy,pandas,scikit-learn,sentence_transformers,umap,plotly,pandera,loguru,tqdm,anthropic,openai

print()
print("ARTIFEX Labs | linktr.ee/artifexlabs | github.com/tuesdaythe13th")
print("© 2026 ARTIFEX Labs. All Rights Reserved.")